In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Use the root directory of uPHATE repo
import sys
sys.path.append("../")

from SERGIO.SERGIO.sergio import sergio

# For compatibility with sergio which uses an old numpy version
np.int = int
np.float = float

SERGIO_DATASETS_PATH = Path("../SERGIO/data_sets")

In [2]:
sergio_params = pd.DataFrame({
    "DS_ID": [1, 2, 3, 4, 5, 6, 7, 8, 13, 14],
    "number_genes":[100, 400, 1200, 100,  100, 100,  100, 100, 100, 100],
    "number_bins":[9, 9, 9, 9, 9, 9, 9, 9, 4, 6],
    # Noise params
    "Outlier_Genes_pi": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
    "Outlier_Genes_mu": [0.8, 0.8, 0.8, 3.0, 3.0, 5.0, 3.0, 4.5, 0.8, 0.8],
    "Outlier_Genes_sigma": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    "Library_Size_mu": [4.8, 6.0, 7.0, 6.0, 6.0, 4.5, 4.4, 10.8, 3.6, 5.0],
    "Library_Size_sigma": [0.3, 0.4, 0.4, 0.3, 0.4, 0.7, 0.8, 0.55, 0.4, 0.4],
    "Dropouts_k": [20, 12, 8, 8, 8, 8, 8, 2, 8, 4],
    "Dropouts_q": [82, 80, 80, 74, 82, 45, 85, 92, 70, 80],
    "Low_Quality_threshold": [5, 5, 5, 5, 5, 5, 5, 2500, 5, 5]
})

# Simulate Clean Data _ Steady-State Simulation

In [ ]:
datasets = list(SERGIO_DATASETS_PATH.iterdir())
clean_data = {}
noisy_data = {}

for _, params in sergio_params.iterrows():
    id = int(params['DS_ID'])
    matches = [p for p in datasets if f"DS{id}" in p.name]
    if not matches:
        continue
    dataset = matches[0]

    if id == 3:
        # Takes too long
        print(f"### Skipping {dataset.name}")
        continue

    print(f"----- Simulating {dataset.name} -----")

    dynamics = "dynamics" in dataset.name
    num = dataset.name.split("_")[-2]
    number_genes = dataset.name.split("G_")[0].split("_")[1]
    number_bins = int(dataset.name.split("T_")[0].split("_")[-1])

    if dynamics:
        df = pd.read_csv(
            dataset / f"bMat_cID{num}.tab", sep="\t", header=None, index_col=None
        )
        bMat = df.values
        noise_params = 0.3
        noise_params_splice = 0.07
    else:
        bMat = None
        noise_params = 1.0
        noise_params_splice = None

    sim = sergio(
        number_genes=int(number_genes),
        number_bins=number_bins,
        number_sc=300,
        noise_params=noise_params,
        noise_params_splice=noise_params_splice,
        decays=0.8,
        sampling_state=15,
        noise_type="dpd",
        dynamics=dynamics,
        bifurcation_matrix=bMat,
    )

    sim.build_graph(
        input_file_taregts=dataset / f"Interaction_cID_{num}.txt",
        input_file_regs=dataset / f"Regs_cID_{num}.txt",
        shared_coop_state=2,
    )
    if dynamics:
        df = pd.read_csv(
            dataset / f"bMat_cID{num}.tab", sep="\t", header=None, index_col=None
        )
        bMat = df.values
        sim.simulate_dynamics()
        exprU, exprS = sim.getExpressions_dynamics()
        exprU_clean = np.concatenate(exprU, axis=1)
        exprS_clean = np.concatenate(exprS, axis=1)
        clean_data[id] = (exprU_clean, exprS_clean)
        noisy_data[id] = (exprU, exprS)
    else:
        sim.simulate()
        expr = sim.getExpressions()
        expr_clean = np.concatenate(expr, axis=1)
        clean_data[id] = expr_clean
        noisy_data[id] = expr

In [ ]:
SAVE_LOC = Path("../data/sergio")
SAVE_LOC.mkdir(exist_ok=True)
for id in clean_data:
    np.save(SAVE_LOC / f"clean_{id}", clean_data[id])

# Add Technical Noise _ Steady-State Simulations

In [ ]:
def add_noise_steady_state(expr, params):
    expr_O = sim.outlier_effect(
        expr,
        outlier_prob=params["Outlier_Genes_pi"],
        mean=params["Outlier_Genes_mu"],
        scale=params["Outlier_Genes_sigma"],
    )
    libFactor, expr_O_L = sim.lib_size_effect(
        expr_O, mean=params["Library_Size_mu"], scale=params["Library_Size_sigma"]
    )
    binary_ind = sim.dropout_indicator(
        expr_O_L, shape=params["Dropouts_k"], percentile=params["Dropouts_q"]
    )
    expr_O_L_D = np.multiply(binary_ind, expr_O_L)
    count_matrix = sim.convert_to_UMIcounts(expr_O_L_D)
    count_matrix = np.concatenate(count_matrix, axis=1)
    return count_matrix


def add_noise_dynamics(exprU, exprS, params):
    exprU_O, exprS_O = sim.outlier_effect_dynamics(
        exprU,
        exprS,
        outlier_prob=params["Outlier_Genes_pi"],
        mean=params["Outlier_Genes_mu"],
        scale=params["Outlier_Genes_sigma"],
    )
    libFactor, exprU_O_L, exprS_O_L = sim.lib_size_effect_dynamics(
        exprU_O,
        exprS_O,
        mean=params["Library_Size_mu"],
        scale=params["Library_Size_sigma"],
    )
    binary_indU, binary_indS = sim.dropout_indicator_dynamics(
        exprU_O_L,
        exprS_O_L,
        shape=params["Dropouts_k"],
        percentile=params["Dropouts_q"],
    )
    exprU_O_L_D = np.multiply(binary_indU, exprU_O_L)
    exprS_O_L_D = np.multiply(binary_indS, exprS_O_L)
    count_matrix_U, count_matrix_S = sim.convert_to_UMIcounts_dynamics(
        exprU_O_L_D, exprS_O_L_D
    )
    count_matrix_U = np.concatenate(count_matrix_U, axis=1)
    count_matrix_S = np.concatenate(count_matrix_S, axis=1)
    return count_matrix_U, count_matrix_S

In [ ]:
noisy_data = {}

for _, params in sergio_params.iterrows():
    id = int(params["DS_ID"])
    matches = [p for p in datasets if f"DS{id}" in p.name]
    if not matches:
        continue
    dataset = matches[0]
    if id not in clean_data:
        print(f"### Skipping {id}")

    print(f"----- Noising {dataset.name} -----")

    dynamics = "dynamics" in dataset.name

    if dynamics:
        exprU, exprS =  noisy_data[id]
        noisy_data[id] = add_noise_dynamics(exprU, exprS, params)
    else:
        expr = noisy_data[id]
        noisy_data[id] = add_noise_steady_state(expr, params)

----- Noising De-noised_100G_9T_300cPerT_4_DS1 -----


AxisError: axis 1 is out of bounds for array of dimension 1

In [ ]:
for id in noisy_data:
    np.save(SAVE_LOC / f"noisy_{id}", noisy_data[id])